In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load Data
df = pd.read_csv('Family_Income_and_Expenditure.csv')

# Create target labels based on Income
quintile_labels = ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (Highest)']
df['income_quintile'] = pd.qcut(df['Total Household Income'], 5, labels=quintile_labels)
le = LabelEncoder()
df['labels'] = le.fit_transform(df['income_quintile'])

# Vectorized function for faster execution and richer text representation
def generate_rich_fies_text(dataframe):
    return (
        "Household of " + dataframe['Total Number of Family members'].astype(str) + " members " +
        "(" + dataframe['Total number of family members employed'].astype(str) + " employed). " +
        "Type: " + dataframe['Type of Building/House'].astype(str) + ", " +
        "Roof: " + dataframe['Type of Roof'].astype(str) + ", " +
        "Walls: " + dataframe['Type of Walls'].astype(str) + ". " +
        "House has " + dataframe['Number of bedrooms'].astype(str) + " bedrooms. " +
        "Amenities: Electricity=" + dataframe['Electricity'].astype(str) +
        ", TV=" + dataframe['Number of Television'].astype(str) +
        ", Aircon=" + dataframe['Number of Airconditioner'].astype(str) +
        ", Car/Jeep=" + dataframe['Number of Car, Jeep, Van'].astype(str) + ". " +
        "Expenditures: Food: " + dataframe['Total Food Expenditure'].map('{:,.0f}'.format) + " PHP, " +
        "Housing/Water: " + dataframe['Housing and water Expenditure'].map('{:,.0f}'.format) + " PHP, " +
        "Edu: " + dataframe['Education Expenditure'].map('{:,.0f}'.format) + " PHP, " +
        "Trans: " + dataframe['Transportation Expenditure'].map('{:,.0f}'.format) + " PHP, " +
        "Medical: " + dataframe['Medical Care Expenditure'].map('{:,.0f}'.format) + " PHP, " +
        "Communication: " + dataframe['Communication Expenditure'].map('{:,.0f}'.format) + " PHP."
    )

print("Generating rich context text strings...")
df['text_input'] = generate_rich_fies_text(df)

# Split datasets
train_subset = df.sample(20000, random_state=1234)
test_subset = df.drop(train_subset.index).sample(4000, random_state=1234)

train_ds = Dataset.from_pandas(train_subset[['text_input', 'labels']])
test_ds = Dataset.from_pandas(test_subset[['text_input', 'labels']])

# Tokenization
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_fn(batch):
    return tokenizer(batch["text_input"], padding="max_length", truncation=True, max_length=256)

train_tokenized = train_ds.map(tokenize_fn, batched=True)
test_tokenized = test_ds.map(tokenize_fn, batched=True)

# Model Setup
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=5)

args = TrainingArguments(
    output_dir="./fies_ai_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, p.predictions.argmax(-1))}
)

print("Starting training with enriched dataset features...")
trainer.train()

# 1. Get raw predictions on the test set
predictions_output = trainer.predict(test_tokenized)
y_pred = predictions_output.predictions.argmax(-1)
y_true = predictions_output.label_ids

# 2. Print Classification Report
print("\n--- Detailed Performance Report ---")
print(classification_report(y_true, y_pred, target_names=quintile_labels))

# 3. Generate Confusion Matrix
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=[f"True {l}" for l in quintile_labels],
                         columns=[f"Pred {l}" for l in quintile_labels])
print(cm_df)

# --- CONSULTANT FUNCTION ---
def ai_financial_consultant(members, employed, bldg_type, roof, walls, bedrooms,
                            electricity, tv, aircon, car, food, housing, edu, transport, medical, comm):

    # 1. Format text
    query = (f"Household of {members} members ({employed} employed). "
             f"Type: {bldg_type}, Roof: {roof}, Walls: {walls}. "
             f"House has {bedrooms} bedrooms. "
             f"Amenities: Electricity={electricity}, TV={tv}, Aircon={aircon}, Car/Jeep={car}. "
             f"Expenditures: Food: {food:,} PHP, Housing/Water: {housing:,} PHP, "
             f"Edu: {edu:,} PHP, Trans: {transport:,} PHP, Medical: {medical:,} PHP, Communication: {comm:,} PHP.")

    # 2. Tokenize and pass to device
    inputs = tokenizer(query, return_tensors="pt", padding=True, truncation=True, max_length=256).to(model.device)

    # 3. Predict
    with torch.no_grad():
        logits = model(**inputs).logits

    pred_id = torch.argmax(logits, dim=1).item()
    quintile = le.inverse_transform([pred_id])[0]

    return {
        "Input Data Sent to AI": query,
        "AI Predicted Quintile": quintile,
    }

Generating rich context text strings...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training with enriched dataset features...


... (5 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



--- Detailed Performance Report ---
              precision    recall  f1-score   support

 Q1 (Lowest)       0.77      0.76      0.77       817
          Q2       0.55      0.65      0.59       820
          Q3       0.52      0.51      0.51       770
          Q4       0.58      0.58      0.58       790
Q5 (Highest)       0.85      0.73      0.78       803

    accuracy                           0.65      4000
   macro avg       0.65      0.64      0.65      4000
weighted avg       0.66      0.65      0.65      4000


--- Confusion Matrix ---
                   Pred Q1 (Lowest)  Pred Q2  Pred Q3  Pred Q4  \
True Q1 (Lowest)                625      177       14        1   
True Q2                         147      532      132        9   
True Q3                          34      213      390      131   
True Q4                           7       48      180      457   
True Q5 (Highest)                 1        6       29      184   

                   Pred Q5 (Highest)  
True Q1 (Low

In [ ]:
# Save the model
model.save_pretrained("./final_fies_model")
tokenizer.save_pretrained("./final_fies_model")
import joblib
joblib.dump(le, "./final_fies_model/label_encoder.pkl")

In [ ]:
print("\n" + "="*40)
print("TESTING THE MODEL WITH CUSTOM VALUES")
print("="*40)

print("\n--- Test 1: Low Spending Profile (Expected Q1/Q2) ---")
result_1 = ai_financial_consultant(
    members=5, employed=1, bldg_type="Single house",
    roof="Light material (cogon,nipa,anahaw)", walls="Light", bedrooms=1,
    electricity=0, tv=0, aircon=0, car=0,
    food=8000, housing=1500, edu=500, transport=1200, medical=200, comm=100
)
for k, v in result_1.items():
    print(f"{k}: {v}")

print("\n--- Test 2: High Spending Profile (Expected Q5) ---")
result_2 = ai_financial_consultant(
    members=3, employed=2, bldg_type="Single house",
    roof="Strong material (galvanized iron,alu...)", walls="Strong", bedrooms=4,
    electricity=1, tv=2, aircon=3, car=1,
    food=45000, housing=35000, edu=40000, transport=25000, medical=8000, comm=6000
)
for k, v in result_2.items():
    print(f"{k}: {v}")


TESTING THE MODEL WITH CUSTOM VALUES

--- Test 1: Low Spending Profile (Expected Q1/Q2) ---
Input Data Sent to AI: Household of 5 members (1 employed). Type: Single house, Roof: Light material (cogon,nipa,anahaw), Walls: Light. House has 1 bedrooms. Amenities: Electricity=0, TV=0, Aircon=0, Car/Jeep=0. Expenditures: Food: 8,000 PHP, Housing/Water: 1,500 PHP, Edu: 500 PHP, Trans: 1,200 PHP, Medical: 200 PHP, Communication: 100 PHP.
AI Predicted Quintile: Q1 (Lowest)

--- Test 2: High Spending Profile (Expected Q5) ---
Input Data Sent to AI: Household of 3 members (2 employed). Type: Single house, Roof: Strong material (galvanized iron,alu...), Walls: Strong. House has 4 bedrooms. Amenities: Electricity=1, TV=2, Aircon=3, Car/Jeep=1. Expenditures: Food: 45,000 PHP, Housing/Water: 35,000 PHP, Edu: 40,000 PHP, Trans: 25,000 PHP, Medical: 8,000 PHP, Communication: 6,000 PHP.
AI Predicted Quintile: Q5 (Highest)
